# Sesión 3 - Ejercicios: ETL con Polars

Trabajamos con cinco tablas de un registro escolar ficticio guardadas en `test_data.xlsx`. Cada ejercicio construye sobre el anterior, así que conviene ejecutarlos en orden.

## Ejercicio 1 - Carga desde Excel

Carga cada hoja del archivo `test_data.xlsx` como un DataFrame de Polars. Nombra cada variable `df_<nombre_de_la_hoja>`.

In [ ]:
import polars as pl
# NO OLVIDAR INSTALAR pip install fastexcel
ruta = "test_data.xlsx"

df_alumnos = pl.read_excel(ruta, sheet_name="alumnos")
df_notas = pl.read_excel(ruta, sheet_name="notas")
df_asistencia = pl.read_excel(ruta, sheet_name="asistencia")
df_profesores = pl.read_excel(ruta, sheet_name="profesores")
df_asigna_clase = pl.read_excel(ruta, sheet_name="asigna_clase")


In [2]:
df_alumnos.head()

cod_estudiante,nombre,sexo,edad,pais,ciudad,altura,paralelo,anio_nac,mes_nac,dia_nac
i64,str,str,i64,str,str,f64,str,i64,i64,i64
1,"""Raúl Martos""","""f""",53,"""Ecuador""","""Quito""",174.384169,"""A""",1973,4,21
2,"""Lazaro Herrera""","""m""",58,"""Ecuador""","""Guayaquil""",161.831894,"""A""",1968,7,29
3,"""Patricio Sacristan""","""m""",59,"""Ecuador""","""Ambato""",174.921596,"""A""",1967,6,12
4,"""Angelina del Pino""","""f""",40,"""Ecuador""","""Cuenca""",190.579941,"""C""",1986,11,21
5,"""Adriana San-Jose""","""f""",47,"""Ecuador""","""Manta""",181.780877,"""A""",1979,11,8


In [3]:
df_notas.head()

cod_estudiante,nombre,materia,nota
i64,str,str,f64
1,"""Raul Martos""","""CASTELLANO""",0.0
2,"""Lazaro Errera""","""CASTELLANO""",6.887131
3,"""Patricio Sacristan""","""CASTELLANO""",0.449471
4,"""Angelina del Pino""","""CASTELLANO""",1.124459
5,"""Adriana San Jose""","""CASTELLANO""",2.404431


## Ejercicio 2 - Cast de tipo

La columna `sexo` de `df_alumnos` es de tipo `String`. Conviértela al tipo `Categorical`.

In [4]:
df_alumnos = df_alumnos.with_columns(
    pl.col("sexo").cast(pl.Categorical)
)

df_alumnos.select("sexo").unique()


sexo
cat
"""m"""
"""f"""


## Ejercicio 3 - Drop de columna

La tabla `df_asistencia` tiene una columna que contiene el total de días asistidos por alumno. Esa columna la calcularemos nosotros desde los datos: elimínala.

In [5]:
df_asistencia = df_asistencia.drop("total")

# Verificar que ya no aparece
print(df_asistencia.columns)


['cod_estudiante', 'nombre', 'a_19950501', 'a_19950502', 'a_19950503', 'a_19950504', 'a_19950505', 'a_19950508', 'a_19950509', 'a_19950510', 'a_19950511', 'a_19950512', 'a_19950515', 'a_19950516', 'a_19950517', 'a_19950518', 'a_19950519', 'a_19950522', 'a_19950523', 'a_19950524', 'a_19950525', 'a_19950526', 'a_19950529', 'a_19950530', 'a_19950531']


## Ejercicio 4 - Transformación numérica

La columna `altura` de `df_alumnos` está en centímetros. Agrega una columna nueva llamada `altura_m` con la altura en metros, redondeada a dos decimales.

In [6]:
df_alumnos = df_alumnos.with_columns(
    (pl.col("altura") / 100).round(2).alias("altura_m")
)

df_alumnos.select(["nombre", "altura", "altura_m"]).head(5)


nombre,altura,altura_m
str,f64,f64
"""Raúl Martos""",174.384169,1.74
"""Lazaro Herrera""",161.831894,1.62
"""Patricio Sacristan""",174.921596,1.75
"""Angelina del Pino""",190.579941,1.91
"""Adriana San-Jose""",181.780877,1.82


## Ejercicio 5 - Homologar columna `materia`

En `df_notas` las materias están en mayúsculas sin tildes (`MATEMATICAS`). En `df_profesores` están en minúsculas con tildes (`matemáticas`). Un join directo entre estas tablas no encontraría coincidencias.

Normaliza la columna `materia` en ambas tablas: minúsculas, sin espacios al inicio o al final y sin tildes.

Pista: puede usar str.replace_all("á","a") para reemplazar la á por la a

In [7]:
df_notas = df_notas.with_columns(
    (pl.col("materia")
     .str.to_lowercase()
     .str.strip_chars()
     .str.replace_all("á", "a")
     .str.replace_all("é", "e")
     .str.replace_all("í", "i")
     .str.replace_all("ó", "o")
     .str.replace_all("ú", "u")).alias("materia")
)
df_profesores = df_profesores.with_columns(
    (pl.col("materia")
     .str.to_lowercase()
     .str.strip_chars()
     .str.replace_all("á", "a")
     .str.replace_all("é", "e")
     .str.replace_all("í", "i")
     .str.replace_all("ó", "o")
     .str.replace_all("ú", "u")).alias("materia")
)

# Verificar que los valores coinciden ahora
print("Valores en notas:", df_notas["materia"].unique().sort().to_list())
print("Valores en profesores:", df_profesores["materia"].unique().sort().to_list())


Valores en notas: ['castellano', 'educacion fisica', 'matematicas', 'teatro']
Valores en profesores: ['castellano', 'educacion fisica', 'matematicas', 'teatro']


## Ejercicio 6 - Agregación con `group_by`

Calcula la edad media de los alumnos desagregada por sexo. Incluye también el conteo de alumnos por grupo.

In [8]:
df_alumnos.group_by("sexo").agg(
        [pl.col("edad").mean().alias("edad_media"),
         pl.len().alias("n_alumnos")]).sort("sexo")


sexo,edad_media,n_alumnos
cat,f64,u32
"""f""",38.818182,11
"""m""",39.666667,9


## Ejercicio 7 - Join y agregación

Une `df_notas` con `df_alumnos` usando `cod_estudiante` como llave. A partir del resultado, calcula la nota media por paralelo. Incluye el número de calificaciones no nulas usadas en cada promedio.

### Join

In [9]:
df_notas_alumnos = df_notas.join(
    df_alumnos.select(["cod_estudiante", "paralelo"]),
    on="cod_estudiante",
    how="left"
)

df_notas_alumnos.head()

cod_estudiante,nombre,materia,nota,paralelo
i64,str,str,f64,str
1,"""Raul Martos""","""castellano""",0.0,"""A"""
2,"""Lazaro Errera""","""castellano""",6.887131,"""A"""
3,"""Patricio Sacristan""","""castellano""",0.449471,"""A"""
4,"""Angelina del Pino""","""castellano""",1.124459,"""C"""
5,"""Adriana San Jose""","""castellano""",2.404431,"""A"""


### Agregación

In [10]:
df_notas_alumnos.group_by("paralelo").agg([
    pl.col("nota").mean().round(2).alias("nota_media"),
    pl.col("nota").count().alias("n_notas"),
]).sort("paralelo")

paralelo,nota_media,n_notas
str,f64,u32
"""A""",5.95,32
"""B""",6.94,20
"""C""",6.48,20


## Ejercicio 8 - Unpivot, recodificación y agregación

La tabla `df_asistencia` está en formato wide: una columna por día. El valor `X` indica que el alumno asistió; las celdas vacías indican ausencia.

1. Convierte la tabla a formato long con `.unpivot()`.
2. Reemplaza `X` por `1` y los valores nulos por `0` en una nueva columna `asistio` de tipo entero.
3. Convierte las fechas de la columna pivoteada a fehca.
4. Calcula la tasa de asistencia media por alumno (días asistidos / total de días).
5. Agrega una variable booleana `asistencia_ok` que sea `True` si la tasa supera el 80%.

In [11]:
# 1: identificar las columnas de días y hacer el unpivot
asist_long = df_asistencia.unpivot(
    on=pl.selectors.starts_with("a_"),
    index=["cod_estudiante", "nombre"],
    variable_name="fecha",
    value_name="asistio_raw",
)

asist_long.head()

cod_estudiante,nombre,fecha,asistio_raw
i64,str,str,str
1,"""Raúl Martos""","""a_19950501""","""X"""
2,"""Lazaro Herrera""","""a_19950501""","""X"""
3,"""Patricio Sacristan""","""a_19950501""","""X"""
4,"""Angelina del Pino""","""a_19950501""","""X"""
5,"""Adriana San-Jose""","""a_19950501""","""X"""


In [12]:
# 2: recodificar X -> 1, nulo -> 0
asist_long = asist_long.with_columns(
    pl.when(pl.col("asistio_raw") == "X")
      .then(pl.lit(1))
      .otherwise(pl.lit(0))
      .alias("asistio")
)

asist_long.head()

cod_estudiante,nombre,fecha,asistio_raw,asistio
i64,str,str,str,i32
1,"""Raúl Martos""","""a_19950501""","""X""",1
2,"""Lazaro Herrera""","""a_19950501""","""X""",1
3,"""Patricio Sacristan""","""a_19950501""","""X""",1
4,"""Angelina del Pino""","""a_19950501""","""X""",1
5,"""Adriana San-Jose""","""a_19950501""","""X""",1


In [13]:
# 3: pasar a fecha el string de las columnas

asist_long = asist_long.with_columns(
    pl.col("fecha").str.replace_all("^a_","").str.to_date(format="%Y%m%d")
)

In [14]:
# 4 y 5: tasa de asistencia y variable booleana
resumen_asistencia = (
    asist_long
    .group_by(["cod_estudiante", "nombre"])
    .agg([
        pl.col("asistio").sum().alias("dias_asistidos"),
        pl.col("asistio").mean().round(3).alias("tasa_asistencia"),
    ])
    .with_columns(
        (pl.col("tasa_asistencia") > 0.80).alias("asistencia_ok")
    )
    .sort("tasa_asistencia", descending=True)
)


resumen_asistencia

cod_estudiante,nombre,dias_asistidos,tasa_asistencia,asistencia_ok
i64,str,i32,f64,bool
8,"""Gloria Berrocal""",23,1.0,true
14,"""Richard Yañez""",23,1.0,true
2,"""Lazaro Herrera""",21,0.913,true
17,"""Claúdia Cerda""",21,0.913,true
1,"""Raúl Martos""",21,0.913,true
…,…,…,…,…
6,"""Jeronima Velasco""",11,0.478,false
12,"""Matias Recio""",10,0.435,false
16,"""Luciano Candela""",8,0.348,false
